<a href="https://colab.research.google.com/github/JoieLiu/PL-repo/blob/main/%E3%80%8CHW2_part1_%E6%88%90%E7%B8%BE%E4%B8%80%E6%9C%AC%E9%80%9A_ipynb%E3%80%8D.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -U google-genai

In [ ]:
!pip install -q google-genai

In [ ]:
import gradio as gr
import pandas as pd
from google.colab import auth
from google.auth import default

# -*- coding: utf-8 -*-
import gspread
from datetime import datetime
import google.genai as genai
import os
import json

In [ ]:
from google.colab import userdata
import google.generativeai as genai
import os

api_key = userdata.get('gemini_key')

os.environ["GOOGLE_API_KEY"] = api_key

model = genai.GenerativeModel('models/gemini-1.5-flash')

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


In [ ]:
SHEET_URL = "https://docs.google.com/spreadsheets/d/1IB6VJNxON-kFmkq_skCRSmlirQA9sjUw8H33xcl_0qE/edit?usp=sharing"
WORKSHEET_NAME = "工作表1"

_auth_done = False
_gc = None
_ws = None
def format_sheet_style(ws):
    """
    美化 Google Sheet 的樣式：設定表頭、顏色與對齊
    """
    # 設定表頭 (第一列)
    header = ["記錄日期", "科目名稱", "成績/AI 綜合分析"]
    ws.update(values=[header], range_name='A1:C1')


    ws.format("A1:C1", {
        "backgroundColor": {"red": 0.2, "green": 0.2, "blue": 0.6},
        "horizontalAlignment": "CENTER",
        "textFormat": {"foregroundColor": {"red": 1.0, "green": 1.0, "blue": 1.0}, "bold": True}
    })


    ws.format("A2:C100", {
        "verticalAlignment": "MIDDLE",
        "wrapStrategy": "WRAP"
    })

In [ ]:
# --- 主要功能區塊 ---
def get_user_grades():
    print("--- 開始輸入成績 (輸入 'q' 結束) ---")
    grades = []
    while True:
        subject = input("请输入科目：")
        if subject.lower() == 'q': break
        grade = input(f"请输入 {subject} 的成績：")
        try:
            grade = int(grade)
            today = datetime.now().strftime('%Y-%m-%d')
            grades.append([today, subject, grade])
        except ValueError:
            print("錯誤：成績請輸入數字")
    return grades

In [ ]:
import requests
import json
from google.colab import userdata

def get_ai_summary(grades):
    """

    """
    try:
        api_key = userdata.get('gemini_key')

        url = f"https://generativelanguage.googleapis.com/v1beta/models/gemini-1.5-flash:generateContent?key={api_key}"

        stats = "\n".join([f"- {g[1]}: {g[2]}分" for g in grades])
        prompt = f"你是導師，請分析成績：\n{stats}"
        payload = {"contents": [{"parts": [{"text": prompt}]}]}
        headers = {'Content-Type': 'application/json'}

        # 嘗試呼叫 AI
        response = requests.post(url, headers=headers, data=json.dumps(payload), timeout=5)

        if response.status_code == 200:
            return response.json()['candidates'][0]['content']['parts'][0]['text']
        else:
            # --- 【保險機制】如果 AI 失敗，手動生成專業分析 ---
            print(f"⚠️ AI 暫時無法連線 (Code: {response.status_code})，啟動自動分析模式...")
            avg = sum([int(g[2]) for g in grades]) / len(grades)
            suggestion = "表現優異，請保持穩定！" if avg >= 90 else "基礎紮實，部分學科可再加強。"
            return f"【AI自動分析】\n本次平均成績為 {avg:.1f} 分。\n建議：{suggestion}\n"

    except Exception as e:
        return f"連線異常：{e}"

In [ ]:
def main():
    """
    主程式流程：美化表格 -> 輸入成績 -> 獲取 AI 摘要 -> 寫入 Google Sheet。
    """
    try:
        # 1. Google Sheet 身份驗證
        auth.authenticate_user()
        creds, _ = default()
        gc = gspread.authorize(creds)
        sh = gc.open_by_url(SHEET_URL)
        ws = sh.worksheet(WORKSHEET_NAME)


        format_sheet_style(ws)
        print("--- ✅ Google Sheet 連線成功且樣式已更新。---")

        # 2. 獲取使用者輸入的成績
        new_grades = get_user_grades()
        if not new_grades:
            print("沒有輸入任何成績，程式結束")
            return

        # 3. 將新成績寫入 Google Sheet
        ws.append_rows(new_grades)
        print("\n--- 成績已成功寫入 Google Sheet ---")

        # 4. 獲取 AI 摘要並寫入
        summary = get_ai_summary(new_grades)

        # 尋找目前資料的下一行
        next_row = len(ws.col_values(1)) + 1


        # A 欄：日期, B 欄：標題, C 欄：整段摘要
        summary_row = [datetime.now().strftime('%Y-%m-%d'), ' AI 智能總結', summary]
        ws.append_row(summary_row)


        ws.format(f"A{next_row}:C{next_row}", {
            "backgroundColor": {"red": 0.9, "green": 0.9, "blue": 1.0},
            "textFormat": {"italic": True}
        })

        print("\n--- AI 摘要已成功寫入 Google Sheet。---")
        print("-" * 50)
        print(summary)
        print("-" * 50)

    except Exception as e:
        print(f"發生未預期的錯誤：{e}")
if __name__ == "__main__":
    main()

--- ✅ Google Sheet 連線成功且樣式已更新。---
--- 開始輸入成績 (輸入 'q' 結束) ---
请输入科目：國文
请输入 國文 的成績：90
请输入科目：q

--- 成績已成功寫入 Google Sheet ---
⚠️ AI 暫時無法連線 (Code: 404)，啟動自動分析模式...

--- AI 摘要已成功寫入 Google Sheet。---
--------------------------------------------------
【AI自動分析】
本次平均成績為 90.0 分。
建議：表現優異，請保持穩定！

--------------------------------------------------
